## ENVIRONMENT SETUP AND PACKAGE INSTALLATION


 This cell is used to install all the required libraries for the activity. I uninstall first using the

`!pip uninstall -y` command to remove the existying versions of LangChain, LangChain Core, LangChain Community, LangChain Text Splitters, and LangGraph-related packages. This is necessary to prevent the versions conflicts. Newer versions may not support the `langchain.chains` functions that is used later by the model. I used this because I was able to install the newer versions so I have to uninstall which is more compatible with my RAG implementation in order to avoid the error regarding to module not found.

`!pip install -q` command is used to install the specific versions in order for the model to be compatible with RAG implementation.

Specifically `langchain==0.3.27` This version is older and later the pipeline also uses `create_retrieval_chain` and `create_stuff_documents_chain` which is not compatible with the newer version.`langchain` provides the main framework for connecting the different components of the RAG pipeline.

`langchain-core>=0.3.72,<1.0.0"` provides its fundamental components that is required by the installed LangChain packages.

`langchain-community==0.3.27` provides the DirectoryLoader used to load the PDF files that is stored in my folder which is my_data/ folder.

`langchain-text-splitters` provides the RecursiveCharacterTextSplitter which is used to divide the loaded documents into smaller text chunks, before they are converted into embeddings, making the information easier to retrieve.

`langchain-huggingface` connects the system to a Hugging Face embedding model, specifically all-MiniLM-L6-v2, which converts the document chunks into numerical vector representations.

`chromaDB` serves as the vector database that stores these embeddings and is used by the retriever to compare the user's query with the stored document chunks and return the top 3 most relevant chunks (k=3).

`langchain-groq` connects LangChain to the Groq-hosted LLM, which generates the final response using the retrieved information. The retrieved top three chunks are inserted into the {context} of the system prompt, while the user's question is passed through {input}, allowing the LLM to generate an answer based on the retrieved document information.

`pypdf` and `unstructured[pdf]` provide the necessary PDF-processing functionality for extracting content from the uploaded PDF documents.

In [ ]:
# Cell 1: Environment Setup & Package Installation

# Remove conflicting newer LangChain/LangGraph packages
!pip uninstall -y \
    langchain \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    langchain-classic \
    langgraph \
    langgraph-sdk \
    langgraph-prebuilt \
    langgraph-checkpoint

# Install versions compatible with the notebook
!pip install -q \
    "langchain==0.3.27" \
    "langchain-core>=0.3.72,<1.0.0" \
    "langchain-community==0.3.27" \
    "langchain-text-splitters==0.3.9" \
    "langchain-groq==0.3.7" \
    "langchain-huggingface==0.3.1" \
    chromadb \
    pypdf \
    "unstructured[pdf]"

Found existing installation: langchain 1.3.17
Uninstalling langchain-1.3.17:
  Successfully uninstalled langchain-1.3.17
Found existing installation: langchain-core 1.6.0
Uninstalling langchain-core-1.6.0:
  Successfully uninstalled langchain-core-1.6.0
Found existing installation: langgraph 1.2.11
Uninstalling langgraph-1.2.11:
  Successfully uninstalled langgraph-1.2.11
Found existing installation: langgraph-sdk 0.4.3
Uninstalling langgraph-sdk-0.4.3:
  Successfully uninstalled langgraph-sdk-0.4.3
Found existing installation: langgraph-prebuilt 1.1.0
Uninstalling langgraph-prebuilt-1.1.0:
  Successfully uninstalled langgraph-prebuilt-1.1.0
Found existing installation: langgraph-checkpoint 4.2.0
Uninstalling langgraph-checkpoint-4.2.0:
  Successfully uninstalled langgraph-checkpoint-4.2.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 42.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... do

## API KEY CONFIGURATION

This cell is used to securely configure the Groq API key that will be used later by the model to access the Groq-hosted LLM.

`import os` is used to import the Python os module. In this implementation, the os module is specifically used to access and store the GROQ_API_KEY inside the Google Colab runtime's environment variables. This allows the API key to be accessed later by ChatGroq without directly writing the actual API key in the notebook.

`import getpass` is used to import the getpass module, which provides a secure way to enter the Groq API key. Unlike a normal input() function, getpass.getpass() does not display the API key as plain text while I am typing it, which prevents the key from being exposed in the notebook.

`if "GROQ_API_KEY" not in os.environ:` is used to check whether the GROQ_API_KEY already exists in the current Google Colab environment. If the API key is already stored in the environment, the code inside the if statement will not run, so I do not have to enter the key again during the same runtime session.

`getpass.getpass("Enter your Groq API Key: ")` prompts me to enter my Groq API key securely. The characters of the API key are hidden while being entered, preventing the key from appearing directly in the notebook.

`os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")` stores the API key that I entered as an environment variable named GROQ_API_KEY. Later in the RAG implementation, ChatGroq uses this API key to authenticate the request to Groq and access the selected LLM. The LLM is then used in the generation part of the RAG pipeline to generate the final answer based on the retrieved document chunks.

This approach is used directly to hide my Groq API key in the code because it prevents the actual API key from being exposed when the Google Colab notebook is viewed or shared.

In [1]:
# Cell 2: Secure API Key Configuration

import os

import getpass

# Prompts for API key securely without saving or echoing plain text in notebook cells

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


## LOADING DATA AND CHUNKING

This cell is used to load the PDF documents from the my_data/ folder and divide their extracted content into smaller chunks. These chunks will later be converted into embeddings and stored in ChromaDB for retrieval.

`from langchain_community.document_loaders import DirectoryLoader` imports the DirectoryLoader from LangChain Community. I used DirectoryLoader to automatically locate and load the documents that I placed inside the my_data/ folder instead of loading each PDF file individually.

`from langchain_text_splitters import RecursiveCharacterTextSplitter` I used this because the content extracted from the PDF documents is too large to efficiently retrieve as a whole. The text splitter divides the loaded documents into smaller chunks that can later be converted into embeddings and searched by the retriever.

`os.makedirs("my_data", exist_ok=True)` creates the my_data directory where I place the PDF files that will serve as the knowledge base of my RAG system.

`loader = DirectoryLoader("my_data/", glob="**/*.*", show_progress=True)` creates the document loader and tells it to look inside the my_data/ folder. The glob="**/*.*" pattern allows the loader to find files inside the directory and its subdirectories, while show_progress=True displays the loading progress while the documents are being processed.

`raw_documents = loader.load()` executes the loader and extracts the content from the files found in the my_data/ folder. The loaded content is stored in the raw_documents variable as LangChain document objects. This are the pdf files that I've uploaded in my folder.

`text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)` configures how the loaded document content will be divided. I set chunk_size=500, which means the splitter aims to create chunks containing up to approximately 500 characters. I also set chunk_overlap=50, which allows approximately 50 characters from the previous chunk to overlap with the next chunk. I used this overlap to help preserve context when related information is located near the boundary between two chunks.

`documents = text_splitter.split_documents(raw_documents)` applies the configured text splitter to the loaded documents. The resulting smaller chunks are stored in the documents variable. These are the chunks that will be passed to the Hugging Face embedding model in the next stage of my RAG implementation and then stored as vector embeddings in ChromaDB.

`print(f"Loaded {len(raw_documents)} raw document(s) and split into {len(documents)} chunks.")` displays the number of raw documents successfully loaded and the total number of chunks created after text splitting.

In [13]:
# Cell 3: Loading Custom Data & Chunking

from langchain_community.document_loaders import DirectoryLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter


# Create target data directory

os.makedirs("my_data", exist_ok=True)


# Load all documents from the directory

loader = DirectoryLoader("my_data/", glob="**/*.*", show_progress=True)

raw_documents = loader.load()


# Split documents into smaller semantic chunks

text_splitter = RecursiveCharacterTextSplitter(

chunk_size=500,

chunk_overlap=50

)

documents = text_splitter.split_documents(raw_documents)


print(f"Loaded {len(raw_documents)} raw document(s) and split into {len(documents)} chunks.")


  0%|          | 0/1 [00:00<?, ?it/s]WARNING:unstructured:No languages specified, defaulting to English.

100%|██████████| 1/1 [00:07<00:00,  7.99s/it]

Loaded 1 raw document(s) and split into 374 chunks.


## EMBEDDING MODEL AND VECTOR DB INDEXING

This cell is used to convert the document chunks created in Cell 3 into numerical vector embeddings, store them in ChromaDB, and configure the vector database as a retriever. This allows the RAG system to search for the document chunks that are most semantically relevant to the user's question.

`from langchain_huggingface import HuggingFaceEmbeddings imports the HuggingFaceEmbeddings` This is used to connect my RAG system to the Hugging Face embedding model that converts the text chunks into numerical vector representations.

`from langchain_community.vectorstores import Chroma imports the Chroma` This is used to store and search the vector embeddings generated from the document chunks. This serves as the vector database of my RAG implementation.

`embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")` This model converts each text chunk from the documents variable into a numerical vector that represents the semantic meaning of the text. I used embeddings because the retriever needs to compare the meaning of the user's query with the meaning of the document chunks rather than relying only on exact keyword matching.

`vectorstore = Chroma.from_documents(documents=documents, embedding=embeddings)` creates the Chroma vector database using the chunks produced in Cell 3. The `documents=documents` parameter provides the document chunks that need to be indexed, while `embedding=embeddings` specifies that all-MiniLM-L6-v2 should be used to convert those chunks into vectors. Chroma stores the generated embeddings together with their corresponding document content and metadata so that the relevant original text can be returned during retrieval.

`retriever = vectorstore.as_retriever(search_kwargs={"k": 3})` converts the Chroma vector store into a retriever that can be used by the RAG pipeline. The search_kwargs={"k": 3} setting tells the retriever to return the top 3 most relevant document chunks for each user's query. When a question is submitted, the same embedding model converts the question into a vector, and the retriever searches Chroma for the three document chunks whose embeddings are most semantically similar to the query.

This process is used in order for the LLM to not search or process all the document chunks for every question. Instead, it will seached and provide only the top 3 relevant chunks that will be use for generating the final answer.

In [14]:
# Cell 4: Embedding Model & Vector DB Indexing

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import Chroma


# Initialize open-source embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


# Store embeddings into Chroma vector database

vectorstore = Chroma.from_documents(

documents=documents,

embedding=embeddings

)


# Set vectorstore as a retriever

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## MODEL INITIALIZATION AND DOMAIN SYSTEM PROMPT

This cell is used to initialize the language model that generates the final response and define the system prompt that controls how the model should answer using the retrieved document chunks. This is the generation component of my RAG implementation.

`from langchain_groq import ChatGroq` This is used to connect my RAG pipeline to a language model hosted through the Groq API using the GROQ_API_KEY that I configured in Cell 2.

`from langchain_core.prompts import ChatPromptTemplate imports ChatPromptTemplate` This is used to structure the instructions, retrieved context, and user's question before they are sent to the language model.

`llm = ChatGroq(model_name="openai/gpt-oss-20b", temperature=2)` initializes the LLM used for response generation. The model_name="openai/gpt-oss-20b" specifies the model that my RAG system will access through Groq. The model that presented on the given code is not compatible with my architecture. So, I used different model to run my activity. The `temperature=0` controls the randomness of the model's generated responses. The lower the temperature the lessen it hallucinates.

`system_prompt` It defines the instructions that the LLM must follow when generating an answer. "You are a specialized AI assistant for the user's uploaded domain." tells the model that its role is to answer questions related to the domain represented by my uploaded documents.

`"Answer questions strictly using ONLY the provided context below."` This restricts the model to the information retrieved from my documents instead of intentionally relying on information outside the provided context. This is important in my RAG implementation because the generated response should be grounded in the retrieved document chunks.

`"If the answer cannot be found in the context, reply: 'I cannot answer based on the provided domain data.'"` It provides a specific fallback response. If the retrieved chunks do not contain enough information to answer the user's question, the model is instructed to state that it cannot answer based on the provided domain data instead of generating an unsupported answer.

`"Context:\n{context}"` It provides the placeholder where the retrieved document chunks will be inserted. In my implementation, the retriever configured in Cell 4 retrieves the top 3 relevant chunks, and these chunks are supplied through {context} before the prompt is sent to the LLM.

`prompt = ChatPromptTemplate.from_messages([...])` It creates the final chat prompt using two message roles. ("system", system_prompt) provides the instructions and retrieved {context} to the model, while ("human", "{input}") represents the user's actual question. The {input} placeholder will be replaced with the query entered by the user when the RAG chain is executed.


In [112]:
# Cell 5: Model Initialization and Domain System Prompt

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate


# Initialize the SLM

llm = ChatGroq(

model_name="openai/gpt-oss-20b",

temperature= 1

)


# Custom domain system prompt

system_prompt = (

"You are a specialized AI assistant for the user's uploaded domain.\n"

"Answer questions strictly using ONLY the provided context below.\n"

"Context:\n{context}"

)


prompt = ChatPromptTemplate.from_messages([

("system", system_prompt),

("human", "{input}"),

])

## PIPELINE ASSEMBLY

This cell is used to assemble the retrieval and generation components into one complete RAG pipeline. It connects the retriever created in Cell 4 with the LLM and prompt configured in Cell 5, allowing the system to retrieve relevant document chunks first and then use those chunks to generate the final answer.

`from langchain.chains import create_retrieval_chain imports the create_retrieval_chain` This is used to connect my retriever with the document-processing chain so that retrieval and response generation can happen as one complete process whenever I submit a query.

`from langchain.chains.combine_documents import create_stuff_documents_chain` This is used to combined the retrieved document chunks with the prompt and send them to the LLM for response generation.

`combine_docs_chain = create_stuff_documents_chain(llm, prompt)` It creates the document-processing chain by connecting the llm and prompt that I initialized in Cell 5. The term "stuff" means that the retrieved document chunks are placed together into the {context} placeholder of my system prompt. The prompt containing this context and the user's {input} is then passed to the LLM so it can generate an answer based on the retrieved information.

`rag_chain = create_retrieval_chain(retriever, combine_docs_chain)` It creates the complete RAG chain by connecting the retriever from Cell 4 with the combine_docs_chain. When I provide a question to this chain, the retriever first searches ChromaDB and returns the top 3 most relevant document chunks (k=3). These retrieved chunks are then passed to combine_docs_chain, inserted into the {context} of the system prompt, and processed together with the user's question by the LLM.

In [113]:
# Cell 6: Pipeline Assembly

from langchain.chains import create_retrieval_chain

from langchain.chains.combine_documents import create_stuff_documents_chain


# Combine prompt and LLM to process context

combine_docs_chain = create_stuff_documents_chain(llm, prompt)


# Assemble full retrieval-augmented generation chain

rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

## TESTING CHATBOT

This cell is used to test the complete RAG pipeline using an in-domain question. It sends a user query to the rag_chain, retrieves the most relevant document chunks, generates an answer from those chunks, and then displays both the answer and the sources of the retrieved chunks.

`user_query = ` This stores the test question that I want to ask the RAG chatbot. It can be anything you like to ask to the Chatbot.

`response = rag_chain.invoke({"input": user_query})` This sends the question to the complete RAG pipeline created in Cell 6. The value of user_query is passed into the {input} placeholder of the prompt. Before the LLM generates the response, the retriever searches the Chroma vector database and retrieves the top 3 document chunks that are most semantically related to the question.

`print("--- DOMAIN QUERY ANSWER ---")` It prints a heading before the generated answer. I

`print(response["answer"])` It accesses the value stored under the "answer" key in the response returned by the RAG chain. This contains the final answer generated by the LLM using the retrieved document chunks and the instructions defined in the system prompt.

`print("\n--- RETRIEVED SOURCE CHUNKS ---")` prints another heading for the retrieved sources.

` for i, doc in enumerate(response["context"]):` loops through the document chunks that were retrieved from ChromaDB. The "context" key contains the chunks selected by the retriever and passed to the LLM. Since the retriever was configured with k=3, this loop normally processes the three most relevant chunks.

`enumerate(response["context"])` provides both the position of each retrieved chunk and the actual document object. The variable i starts from 0, while doc represents the retrieved document chunk.

`print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))` displays the source file associated with each retrieved chunk.

In [114]:
# Cell 7: Testing Your Custom Domain Chatbot


# Test Case 1: In-Domain Query

user_query = " What causes volcanic eruptions? "
response = rag_chain.invoke({"input": user_query})


print("--- DOMAIN QUERY ANSWER ---")

print(response["answer"])


print("\n--- RETRIEVED SOURCE CHUNKS ---")

for i, doc in enumerate(response["context"]):

  print(f"Chunk {i+1} Source:", doc.metadata.get("source", "Unknown"))

--- DOMAIN QUERY ANSWER ---
I’m sorry, but I don’t have enough information from the provided context to answer that question.

--- RETRIEVED SOURCE CHUNKS ---
Chunk 1 Source: my_data/Chapter-5-Study-Skills_p001-p022.pdf
Chunk 2 Source: my_data/Chapter-5-Study-Skills.pdf
Chunk 3 Source: my_data/Chapter-5-Study-Skills_p045-p066.pdf
